# 24 - Agent Evaluation

Agent evaluation is harder than normal RAG evaluation because the system is a **trajectory**.

Evaluate:
- task/goal success
- tool selection
- tool arguments
- tool-call correctness
- number of calls
- failure recovery
- latency
- cost
- safety

Ragas currently documents agent/tool metrics including Tool Call Accuracy, Tool Call F1, and Agent Goal Accuracy.

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU pandas numpy

## 1. Build an agent test set

Each task should include the goal and expected behavior.

In [ ]:
tasks = [
    {
        'id': 'calc-1',
        'input': 'What is 25 * 4?',
        'expected_tools': ['calculator'],
        'expected_answer_contains': '100',
    },
    {
        'id': 'word-1',
        'input': 'Count words in this sentence.',
        'expected_tools': ['word_count'],
        'expected_answer_contains': None,
    },
]
print(tasks)

## 2. Score tool selection

Tool selection is a separate metric from final answer correctness.

In [ ]:
def tool_selection_score(expected_tools, used_tools):
    expected = set(expected_tools)
    used = set(used_tools)
    return int(expected == used)

print(tool_selection_score(['calculator'], ['calculator']))
print(tool_selection_score(['calculator'], ['word_count']))

## 3. Score tool-call F1 conceptually

For multiple expected/used tools, precision and recall can be combined into F1.

In [ ]:
def tool_f1(expected_tools, used_tools):
    expected = set(expected_tools)
    used = set(used_tools)

    tp = len(expected & used)
    precision = tp / len(used) if used else 0.0
    recall = tp / len(expected) if expected else 0.0

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

print(tool_f1(['search', 'calculator'], ['search', 'calculator']))

## 4. Goal accuracy

Goal accuracy asks:
> Did the agent actually complete the user's task?

It is broader than tool selection.

In [ ]:
def goal_accuracy(expected_text, final_answer):
    return int(expected_text.lower() in final_answer.lower())

print(goal_accuracy('100', 'The result is 100.'))

## 5. Trajectory evaluation

A trace can record:

```text
step
tool
arguments
result
duration
error
```

This lets you find unnecessary loops and bad decisions.

In [ ]:
trace = [
    {'step': 1, 'tool': 'calculator', 'duration_ms': 120, 'error': False},
    {'step': 2, 'tool': None, 'duration_ms': 400, 'error': False},
]

print('tool calls:', sum(t['tool'] is not None for t in trace))
print('errors:', sum(t['error'] for t in trace))

## 6. Quality + efficiency

Create a dashboard with:
- success
- tool accuracy
- goal accuracy
- steps
- latency
- cost proxy
- safety failures

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {'agent': 'A', 'success': 0.82, 'tool_f1': 0.90, 'goal_accuracy': 0.84, 'steps': 3.2},
    {'agent': 'B', 'success': 0.86, 'tool_f1': 0.94, 'goal_accuracy': 0.87, 'steps': 2.4},
])

display(df)

## 7. Ragas metrics

Current Ragas documentation includes agent/tool-oriented metrics that can be added to a broader benchmark. Use those alongside application-specific success criteria.

## 8. Evaluation workflow

```text
tasks
 ↓
agent traces
 ↓
automatic scoring
 ↓
manual review of failures
 ↓
fix
 ↓
regression run
```

## 9. Practice

1. Create 30 tasks.
2. Include tool failures.
3. Include adversarial instructions.
4. Measure goal accuracy + tool F1 + steps.
5. Compare two agent versions.

## Evaluation

Do not optimize for one metric. An agent with perfect task success but unsafe tool behavior is not production-ready.

Improvement:
- better tools
- better prompts
- better planning
- better state
- shorter trajectories
- safer permissions.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/